# Sponge Metrics: Urban Flood Susceptibility & SuDS Evaluation
## Grid-Based Framework for Oxford, UK

## Objectives

* Develop a grid-based urban flood susceptibility framework for Oxford, United Kingdom.
* Estimate surface runoff and flood susceptibility under extreme rainfall conditions.
* Evaluate SuDS intervention impacts on runoff reduction and flood resilience.
* Create spatial flood hotspot analysis and climate scenario testing.

## Inputs

* Multi-timescale rainfall forcing system derived from 15-minute observations.
* Event-scale rainfall layers: max_15min_rain, max_1h_rain, storm_event_total_rain.
* Antecedent rainfall layers: rain_3d, rain_5d, rain_7d.
* Extremity indicators: rain_95p_flag, extreme_event_flag.
* Topographic data (elevation, slope per 1 km grid cell).
* Urban surface & land use data (imperviousness %, green space %, building density).
* Floodplain and river proximity flags for spatial context.
* SuDS scenario definitions (rain gardens, permeable paving, detention basins, swales, green roofs).

## Outputs

* Grid-based spatial features DataFrame (elevation, slope, imperviousness, floodplain, river proximity).
* Rainfall forcing tables for event, antecedent, and stress layers.
* Runoff coefficient and effective rainfall outputs.
* Flood susceptibility classifications (Low, Medium, High risk).
* Climate scenario outputs (10-yr, 30-yr, 100-yr rainfall events).
* SuDS comparison maps (existing vs. with interventions).
* Streamlit dashboard visualizations and KPIs.

## Libraries Import

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import math
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("All libraries imported successfully.")


All libraries imported successfully.


---

## Set project root for file I/O

* The notebook should resolve the project root once at startup, then use explicit paths from that root for all reads and writes

The notebook resolves the project root once at startup.
* We access the current directory with Path.cwd()

In [2]:
current_dir = Path.cwd()
for candidate in [current_dir, *current_dir.parents]:
    if (candidate / "data").exists():
        project_root = candidate
        break
else:
    project_root = current_dir

raw_data_dir = project_root / "data" / "raw"
processed_data_dir = project_root / "data" / "processed"
engineered_data_dir = project_root / "data" / "engineered"
current_dir = str(project_root)
current_dir

'c:\\Users\\Home\\OneDrive\\Python Projects\\sponge-metrics'

We want to detect the project root once and keep it fixed for the rest of the notebook.
* os.path.dirname() is no longer needed here because file paths are built explicitly from the project root.
* os.chdir() is not used.

In [3]:
print(f"Project root detected: {project_root}")

Project root detected: c:\Users\Home\OneDrive\Python Projects\sponge-metrics


Confirm the detected project root

In [4]:
current_dir = str(project_root)
current_dir

'c:\\Users\\Home\\OneDrive\\Python Projects\\sponge-metrics'

## Section 1: Spatial Grid Setup

### 1.1 Oxford Grid Cell Definition
A 1 km × 1 km grid framework covering Oxford, United Kingdom was selected to balance spatial representation, computational efficiency, and data availability. Given the city-scale objective of identifying relative surface-water flood susceptibility rather than property-scale inundation, the chosen resolution was considered appropriate.

In [14]:
# Bounding box (Oxford, UK as study area)
lat_min, lat_max = 51.72, 51.80   
lon_min, lon_max = -1.3, -1.2

# Grid size in meters
grid_size_m = 1000  # convert 1 km to m

# Approximate degree step sizes for the target grid size
mean_lat = (lat_min + lat_max) / 2
dlat = grid_size_m / 111320   # 1 degree of latitude is approximately 111.32 km
dlon = grid_size_m / (111320 * math.cos(math.radians(mean_lat)))  

rows = []  # list to hold grid cell information
location_idx = 1 

lat = lat_min  
while lat < lat_max:
    next_lat = min(lat + dlat, lat_max)  

    lon = lon_min
    while lon < lon_max:
        next_lon = min(lon + dlon, lon_max) 

        rows.append(
            {
                "location_id": f"OXF_{location_idx:02d}", 
                "lat_min": round(lat, 6),    # round to 6 decimal places for better readability
                "lat_max": round(next_lat, 6),
                "lon_min": round(lon, 6),
                "lon_max": round(next_lon, 6),
                "centroid_lat": round((lat + next_lat) / 2, 6),
                "centroid_lon": round((lon + next_lon) / 2, 6),
            }
        )

        location_idx += 1     # increment location index for the next grid cell
        lon = next_lon

    lat = next_lat

df = pd.DataFrame(rows) # create DataFrame from the list of grid cells
print(df.shape)  # check the shape of the grid DataFrame to confirm the number of cells created
df.head()  # display the first few rows of the grid DataFrame to verify the output

(63, 7)


,location_id,lat_min,lat_max,lon_min,lon_max,centroid_lat,centroid_lon
0,OXF_01,51.72,51.728983,-1.300000,-1.285487,51.724492,-1.292743
1,OXF_02,51.72,51.728983,-1.285487,-1.270973,51.724492,-1.278230
2,OXF_03,51.72,51.728983,-1.270973,-1.256460,51.724492,-1.263717
3,OXF_04,51.72,51.728983,-1.256460,-1.241947,51.724492,-1.249203
4,OXF_05,51.72,51.728983,-1.241947,-1.227433,51.724492,-1.234690


### 1.2 Topographic Data: Elevation & Slope
Assign elevation per grid cell using information seen on CalcMaps and OS Maps. This is then used to calculate slope (m/m) and derive slope percentage. Slope is used in runoff coefficient model and water accumulation tendency.

In [ ]:
# Assign elevations for the current 9x7 grid
elevations = [
    136, 103, 99, 58, 81, 75, 66,
    117, 102, 59, 58, 60, 64, 78,
    109, 59, 58, 56, 66, 71, 88,
    62, 59, 62, 62, 66, 93, 97,
    58, 57, 67, 61, 68, 102, 103,
    59, 60, 69, 58, 61, 74, 71,
    59, 60, 67, 57, 62, 65, 86,
    59, 74, 69, 58, 58, 96, 115,
    62, 75, 66, 58, 59, 70, 82,
 ]

nrows = 9
ncols = 7

if len(elevations) != len(df):
    raise ValueError(f"Expected {len(df)} elevation values, got {len(elevations)}")

df["elevation_m"] = elevations   # add elevation column (row-major order for current grid)

# Compute actual slope (m/m)
slope = []
for r in range(nrows):
    for c in range(ncols):
        cur = elevations[r * ncols + c]
        diffs = []
        # Left, Right, Up, Down (if present)
        if c > 0:
            diffs.append(abs(cur - elevations[r * ncols + (c - 1)]))  
        if c < ncols - 1:
            diffs.append(abs(cur - elevations[r * ncols + (c + 1)]))
        if r > 0:
            diffs.append(abs(cur - elevations[(r - 1) * ncols + c]))
        if r < nrows - 1:
            diffs.append(abs(cur - elevations[(r + 1) * ncols + c]))

        avg_diff = sum(diffs) / len(diffs)
        slope_val = round(avg_diff / grid_size_m, 4)  # slope in m/m, rounded
        slope.append(slope_val)

df["slope"] = slope
df["slope_pct"] = [round(s * 100, 1) for s in slope]  # add slope percent column

df.head()  # display the first few rows of the final DataFrame with elevations and slopes

,location_id,lat_min,lat_max,lon_min,lon_max,centroid_lat,centroid_lon,elevation_m,slope,slope_pct
0,OXF_01,51.72,51.728983,-1.300000,-1.285487,51.724492,-1.292743,136,0.0260,2.6
1,OXF_02,51.72,51.728983,-1.285487,-1.270973,51.724492,-1.278230,103,0.0127,1.3
2,OXF_03,51.72,51.728983,-1.270973,-1.256460,51.724492,-1.263717,99,0.0283,2.8
3,OXF_04,51.72,51.728983,-1.256460,-1.241947,51.724492,-1.249203,58,0.0213,2.1
4,OXF_05,51.72,51.728983,-1.241947,-1.227433,51.724492,-1.234690,81,0.0167,1.7


### 1.3 Urban Surface Characteristics
Characterize urban surface conditions across the study area using satellite imagery interpretation. Each grid cell was assigned representative imperviousness, vegetation cover, and building-density values based on its dominant land-cover type.

In [85]:
surface_specs = [
    (["OXF_01"], "Suburban Residential", 40, 60, 0.35),
    (["OXF_02", "OXF_60"], "Green Space", 10, 90, 0.05),
    (["OXF_03", "OXF_09", "OXF_17"], "Green Space", 25, 75, 0.15),
    (["OXF_04", "OXF_10", "OXF_55"], "Suburban Residential", 45, 55, 0.45),
    (["OXF_05", "OXF_08", "OXF_12", "OXF_21", "OXF_26", "OXF_44", "OXF_50"], "Dense Residential", 68, 32, 0.68),
    (["OXF_06", "OXF_11", "OXF_13", "OXF_20", "OXF_27", "OXF_28"], "Dense Residential", 72, 28, 0.72),
    (["OXF_07", "OXF_19", "OXF_23", "OXF_34", "OXF_40"], "Dense Residential", 78, 22, 0.78),
    (["OXF_14"], "Urban Core", 80, 20, 0.80),
    (["OXF_15", "OXF_16", "OXF_22"], "Suburban Residential", 60, 40, 0.55),
    (["OXF_18", "OXF_48", "OXF_49"], "Green Space", 8, 92, 0.03),
    (["OXF_24", "OXF_25"], "Urban Core", 88, 12, 0.90),
    (["OXF_29", "OXF_32", "OXF_36", "OXF_39", "OXF_46", "OXF_61"], "Green Space", 12, 88, 0.08),
    (["OXF_30", "OXF_37", "OXF_43", "OXF_53", "OXF_54", "OXF_57"], "Green Space", 25, 75, 0.25),
    (["OXF_31"], "Urban Core", 85, 15, 0.85),
    (["OXF_33", "OXF_35", "OXF_38", "OXF_41", "OXF_42", "OXF_45", "OXF_52"], "Dense Residential", 75, 25, 0.73),
    (["OXF_47", "OXF_51"], "Dense Residential", 67, 33, 0.67),
    (["OXF_56", "OXF_59", "OXF_62", "OXF_63"], "Green Space", 15, 85, 0.15),
    (["OXF_58"], "Suburban Residential", 45, 55, 0.45),
]

surface_lookup = {}
for location_ids, area_type, impervious_surface_pct, green_space_pct, building_density in surface_specs:
    for location_id in location_ids:
        surface_lookup[location_id] = {
            "area_type": area_type,
            "impervious_surface_pct": impervious_surface_pct,
            "green_space_pct": green_space_pct,
            "building_density": building_density,
        }

missing_locations = sorted(set(df["location_id"]) - set(surface_lookup))
if missing_locations:
    raise ValueError(f"Missing surface attributes for: {missing_locations}")

df = df.assign(
    **pd.DataFrame(df["location_id"].map(surface_lookup).tolist(), index=df.index)
)

df.head()

,location_id,lat_min,lat_max,lon_min,lon_max,centroid_lat,centroid_lon,elevation_m,slope,slope_pct,area_type,impervious_surface_pct,green_space_pct,building_density
0,OXF_01,51.72,51.728983,-1.300000,-1.285487,51.724492,-1.292743,136,0.0260,2.6,Suburban Residential,40,60,0.35
1,OXF_02,51.72,51.728983,-1.285487,-1.270973,51.724492,-1.278230,103,0.0127,1.3,Green Space,10,90,0.05
2,OXF_03,51.72,51.728983,-1.270973,-1.256460,51.724492,-1.263717,99,0.0283,2.8,Green Space,25,75,0.15
3,OXF_04,51.72,51.728983,-1.256460,-1.241947,51.724492,-1.249203,58,0.0213,2.1,Suburban Residential,45,55,0.45
4,OXF_05,51.72,51.728983,-1.241947,-1.227433,51.724492,-1.234690,81,0.0167,1.7,Dense Residential,68,32,0.68


### 1.4 Hydrological Features: River Proximity and Floodplain Indicators
River proximity and floodplain indicators were included as supplementary hydrological variables based on visual interpretation from satellite imagery and local floodplain information. Although, they were not incorporated into the final susceptibility model, as the framework focuses specifically on urban surface-water flooding rather than fluvial flooding.

In [86]:
river_proximity = {
    "OXF_04",
    "OXF_05",
    "OXF_11",
    "OXF_12",
    "OXF_17",
    "OXF_18",
    "OXF_22",
    "OXF_23",
    "OXF_24",
    "OXF_25",
    "OXF_29",
    "OXF_30",
    "OXF_32",
    "OXF_36",
    "OXF_37",
    "OXF_38",
    "OXF_39",
    "OXF_43",
    "OXF_44",
    "OXF_45",
    "OXF_46",
    "OXF_50",
    "OXF_51",
    "OXF_52",
    "OXF_53",
    "OXF_54",
    "OXF_58",
    "OXF_59",
    "OXF_60",
    "OXF_61",
}

floodplain_locations = {
    "OXF_18",
    "OXF_29",
    "OXF_30",
    "OXF_32",
    "OXF_36",
    "OXF_37",
    "OXF_39",
    "OXF_43",
    "OXF_46",
    "OXF_53",
    "OXF_54",
    "OXF_59",
    "OXF_60",
    "OXF_61",
}

df["river_proximity_flag"]  = df["location_id"].isin(river_proximity).astype(int)

df["floodplain_flag"] = df["location_id"].isin(floodplain_locations).astype(int)


df.head()

,location_id,lat_min,lat_max,lon_min,lon_max,centroid_lat,centroid_lon,elevation_m,slope,slope_pct,area_type,impervious_surface_pct,green_space_pct,building_density,river_proximity_flag,floodplain_flag
0,OXF_01,51.72,51.728983,-1.300000,-1.285487,51.724492,-1.292743,136,0.0260,2.6,Suburban Residential,40,60,0.35,0,0
1,OXF_02,51.72,51.728983,-1.285487,-1.270973,51.724492,-1.278230,103,0.0127,1.3,Green Space,10,90,0.05,0,0
2,OXF_03,51.72,51.728983,-1.270973,-1.256460,51.724492,-1.263717,99,0.0283,2.8,Green Space,25,75,0.15,0,0
3,OXF_04,51.72,51.728983,-1.256460,-1.241947,51.724492,-1.249203,58,0.0213,2.1,Suburban Residential,45,55,0.45,1,0
4,OXF_05,51.72,51.728983,-1.241947,-1.227433,51.724492,-1.234690,81,0.0167,1.7,Dense Residential,68,32,0.68,1,0


### 1.5 Hydrologic Soil Group (HSG)
Hydrologic Soil Groups were approximated using broad geological and environmental characteristics across Oxford. Cells located near river corridors were assigned HSG-B to represent relatively free-draining alluvial soils, while the remaining urban areas were assigned HSG-D to reflect the low-permeability clay-dominated soils common across much of Oxford. These classifications were later used in deriving Curve Number (CN) values for runoff estimation.

In [ ]:
# Assign Hydrologic Soil Group (HSG) based on river proximity
df["hydrologic_soil_group"] = df["river_proximity_flag"].apply(
    lambda x: "B" if x == 1 else "D"   # assign HSG B if river is nearby, otherwise D
)

df.head()


,location_id,lat_min,lat_max,lon_min,lon_max,centroid_lat,centroid_lon,elevation_m,slope,slope_pct,area_type,impervious_surface_pct,green_space_pct,building_density,river_proximity_flag,floodplain_flag,hydrologic_soil_group
0,OXF_01,51.72,51.728983,-1.300000,-1.285487,51.724492,-1.292743,136,0.0260,2.6,Suburban Residential,40,60,0.35,0,0,D
1,OXF_02,51.72,51.728983,-1.285487,-1.270973,51.724492,-1.278230,103,0.0127,1.3,Green Space,10,90,0.05,0,0,D
2,OXF_03,51.72,51.728983,-1.270973,-1.256460,51.724492,-1.263717,99,0.0283,2.8,Green Space,25,75,0.15,0,0,D
3,OXF_04,51.72,51.728983,-1.256460,-1.241947,51.724492,-1.249203,58,0.0213,2.1,Suburban Residential,45,55,0.45,1,0,B
4,OXF_05,51.72,51.728983,-1.241947,-1.227433,51.724492,-1.234690,81,0.0167,1.7,Dense Residential,68,32,0.68,1,0,B


### 1.6 Spatial Features Output

In [88]:
raw_data_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(raw_data_dir / "oxf_grid_features.csv", index=False)
print(f"✓ Saved: {raw_data_dir / 'oxf_grid_features.csv'}")

✓ Saved: c:\Users\Home\OneDrive\Python Projects\sponge-metrics\data\raw\oxf_grid_features.csv
